In [10]:
import pandas as pd
import random
import numpy as np

df = pd.read_csv('Metalforte_limpa.csv', sep=';', encoding='latin-1')

df.head()

,ID_Registro,Data_Manutencao,Numero_OS,Maquina,Setor,Fabricante,Tipo_Manutencao,Especialidade_x,Horas_Parada,Custo_Manutencao,Peca_Utilizada,Quantidade_Pecas,Causa_Falha,Status_OS,ID_Tecnico,Nome_Tecnico,Especialidade_y
0,1,2025-04-26 00:00:00,OS-1001,Prensa Hidráulica,Montagem,Bosch,Preventiva,Mecânico,17:42,"701,11",Bomba,3,Falha elétrica,Aberta,TEC001,Carlos Silva,Mecânico
1,2,2025-02-17 00:00:00,OS-1002,Prensa Hidráulica,Montagem,Siemens,Preditiva,Automação,17:12,"11805,37",Bomba,7,Falha elétrica,Concluída,TEC004,Fernanda Alves,Eletricista
2,3,2025-03-02 00:00:00,OS-1003,Torno CNC,Montagem,WEG,Preditiva,Automação,08:00,"9588,87",Filtro,8,Lubrificação,Em andamento,TEC002,João Santos,Mecânico
3,4,2025-04-29 00:00:00,OS-1004,Torno CNC,Usin.,Schneider,Preditiva,Automação,03:24,"6849,24",Sensor,9,Falha mecânica,Em andamento,TEC007,Rafael Lima,Mecânico
4,5,2025-06-02 00:00:00,OS-1005,Prensa Hidráulica,Usin.,Siemens,Preditiva,Mecânico,07:36,"12967,13",Sensor,4,Falha mecânica,Concluída,TEC006,Marcos Rocha,Automação


In [11]:

causas_falha = df['Causa_Falha'].unique()

causas_falha


array(['Falha elétrica', 'Lubrificação', 'Falha mecânica',
       'Erro operador', 'Desgaste'], dtype=object)

In [12]:

df['ID_Tecnico'] = np.nan
df['Nome_Tecnico'] = np.nan
df['Especialidade_y'] = np.nan

In [13]:
df.head()

,ID_Registro,Data_Manutencao,Numero_OS,Maquina,Setor,Fabricante,Tipo_Manutencao,Especialidade_x,Horas_Parada,Custo_Manutencao,Peca_Utilizada,Quantidade_Pecas,Causa_Falha,Status_OS,ID_Tecnico,Nome_Tecnico,Especialidade_y
0,1,2025-04-26 00:00:00,OS-1001,Prensa Hidráulica,Montagem,Bosch,Preventiva,Mecânico,17:42,"701,11",Bomba,3,Falha elétrica,Aberta,NaN,NaN,NaN
1,2,2025-02-17 00:00:00,OS-1002,Prensa Hidráulica,Montagem,Siemens,Preditiva,Automação,17:12,"11805,37",Bomba,7,Falha elétrica,Concluída,NaN,NaN,NaN
2,3,2025-03-02 00:00:00,OS-1003,Torno CNC,Montagem,WEG,Preditiva,Automação,08:00,"9588,87",Filtro,8,Lubrificação,Em andamento,NaN,NaN,NaN
3,4,2025-04-29 00:00:00,OS-1004,Torno CNC,Usin.,Schneider,Preditiva,Automação,03:24,"6849,24",Sensor,9,Falha mecânica,Em andamento,NaN,NaN,NaN
4,5,2025-06-02 00:00:00,OS-1005,Prensa Hidráulica,Usin.,Siemens,Preditiva,Mecânico,07:36,"12967,13",Sensor,4,Falha mecânica,Concluída,NaN,NaN,NaN


In [15]:
df.to_csv('dados_atualizado.csv', index=False)

In [ ]:
df = pd.read_csv('dados_atualizado.csv', sep=';', encoding='latin-1')

In [ ]:
# 1. Carrega os arquivos CSV
df = pd.read_csv('Metalforte_limpa.csv', sep=';', encoding='latin-1')
df_tecnicos = pd.read_csv('tabela_tecnicos.csv', sep=';', encoding='latin-1')

# 2. BLINDAGEM: Remove espaços extras e caracteres ocultos (BOM) dos nomes das colunas
df.columns = [col.strip().replace('\ufeff', '').replace('\xef\xbb\xbf', '') for col in df.columns]
df_tecnicos.columns = [col.strip().replace('\ufeff', '').replace('\xef\xbb\xbf', '') for col in df_tecnicos.columns]

# Padroniza a coluna do DataFrame principal (troca '_' por espaço se houver)
df['Causa_Falha'] = df['Causa_Falha'].str.replace('_', ' ').str.strip()

# 3. Dicionário de mapeamento atualizado (usando espaços para garantir o match)
mapeamento_falhas = {
    'Falha elétrica': ['Eletricista'],
    'Falha mecânica': ['Mecânico'],
    'Lubrificação': ['Automação'],
    'Erro operador': ['Eletricista', 'Mecânico', 'Automação'],
    'Desgaste': ['Automação', 'Mecânico']
}

# 4. Função que busca e sorteia o técnico de forma segura
def alocar_tecnico(causa_falha):
    if pd.isna(causa_falha):
        return pd.Series([None, None, None])
    
    causa_falha = str(causa_falha).strip()
    
    if causa_falha not in mapeamento_falhas:
        return pd.Series([None, None, None])
    
    especialidades_permitidas = mapeamento_falhas[causa_falha]
    
    # Filtra os técnicos do segundo CSV
    tecnicos_validos = df_tecnicos[df_tecnicos['Especialidade'].isin(especialidades_permitidas)]
    
    if tecnicos_validos.empty:
        return pd.Series([None, None, None])
    
    # Sorteia um técnico válido
    tecnico_escolhido = tecnicos_validos.sample(n=1).iloc[0]
    
    # Retorna os dados usando as colunas que agora estão 100% limpas
    return pd.Series([
        tecnico_escolhido['ID_Tecnico'], 
        tecnico_escolhido['Nome_Tecnico'], 
        tecnico_escolhido['Especialidade']
    ])

# 5. Aplica a função criando/atualizando as colunas no DataFrame principal
df[['ID_Tecnico', 'Nome_Tecnico', 'Especialidade_y']] = df['Causa_Falha'].apply(alocar_tecnico)

# 6. Mostra o resultado das colunas alteradas
print(df[['Causa_Falha', 'ID_Tecnico', 'Nome_Tecnico', 'Especialidade_y']].head())

      Causa_Falha ID_Tecnico    Nome_Tecnico Especialidade_y
0  Falha elétrica     TEC011  Patricia Souza     Eletricista
1  Falha elétrica     TEC008   Bruno Martins     Eletricista
2    Lubrificação       None            None            None
3  Falha mecânica       None            None            None
4  Falha mecânica       None            None            None


In [19]:
df

,ID_Registro,Data_Manutencao,Numero_OS,Maquina,Setor,Fabricante,Tipo_Manutencao,Especialidade_x,Horas_Parada,Custo_Manutencao,Peca_Utilizada,Quantidade_Pecas,Causa_Falha,Status_OS,ID_Tecnico,Nome_Tecnico,Especialidade_y
0,1,2025-04-26 00:00:00,OS-1001,Prensa Hidráulica,Montagem,Bosch,Preventiva,Mecânico,17:42,"701,11",Bomba,3,Falha elétrica,Aberta,TEC011,Patricia Souza,Eletricista
1,2,2025-02-17 00:00:00,OS-1002,Prensa Hidráulica,Montagem,Siemens,Preditiva,Automação,17:12,"11805,37",Bomba,7,Falha elétrica,Concluída,TEC008,Bruno Martins,Eletricista
2,3,2025-03-02 00:00:00,OS-1003,Torno CNC,Montagem,WEG,Preditiva,Automação,08:00,"9588,87",Filtro,8,Lubrificação,Em andamento,None,None,None
3,4,2025-04-29 00:00:00,OS-1004,Torno CNC,Usin.,Schneider,Preditiva,Automação,03:24,"6849,24",Sensor,9,Falha mecânica,Em andamento,None,None,None
4,5,2025-06-02 00:00:00,OS-1005,Prensa Hidráulica,Usin.,Siemens,Preditiva,Mecânico,07:36,"12967,13",Sensor,4,Falha mecânica,Concluída,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,296,2025-01-03 00:00:00,OS-1296,Fresadora CNC,Montagem,WEG,Preventiva,Mecânico,04:30,"5389,72",Correia,6,Falha elétrica,Concluída,TEC004,Fernanda Alves,Eletricista
296,297,2025-01-13 00:00:00,OS-1297,Torno CNC,Usin.,Bosch,Preventiva,Automação,16:12,"7589,6",Filtro,10,Lubrificação,Em andamento,None,None,None
297,298,2025-01-16 00:00:00,OS-1298,Torno CNC,Montagem,Schneider,Corretiva,Mecânico,01:12,"5692,44",CLP,8,Desgaste,Concluída,None,None,None
298,299,2025-02-10 00:00:00,OS-1299,Corte Laser,Montagem,Bosch,Corretiva,Automação,08:30,"1756,09",Bomba,6,Lubrificação,Aberta,None,None,None
